In [1]:
!pip install streamlit yfinance requests gtts moviepy==1.0.3 pillow numpy pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 28.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [2]:
%%writefile stock_data.py
import yfinance as yf

def get_stock_data(ticker, exchange="NSE"):
    suffix = ".NS" if exchange == "NSE" else ".BO"
    symbol = ticker.upper() + suffix
    try:
        stock = yf.Ticker(symbol)
        info = stock.info
        price = info.get("currentPrice") or info.get("regularMarketPrice")
        return {
            "name": info.get("longName") or ticker,
            "price": round(float(price), 2) if price else "N/A",
            "open": round(float(info.get("open") or 0), 2),
            "high": round(float(info.get("dayHigh") or 0), 2),
            "low": round(float(info.get("dayLow") or 0), 2),
            "volume": int(info.get("volume") or 0),
            "prev_close": round(float(info.get("previousClose") or 0), 2),
            "pe_ratio": info.get("trailingPE", "N/A"),
        }
    except Exception as e:
        return {"name": ticker, "price": "N/A", "open": "N/A",
                "high": "N/A", "low": "N/A", "volume": 0,
                "prev_close": "N/A", "pe_ratio": "N/A"}

Writing stock_data.py


In [3]:
%%writefile news_fetcher.py
import requests
import xml.etree.ElementTree as ET

def get_news(company, max_articles=5):
    try:
        rss_url = "https://economictimes.indiatimes.com/rssfeedstopstories.cms"
        resp = requests.get(rss_url, timeout=8, headers={"User-Agent": "Mozilla/5.0"})
        root = ET.fromstring(resp.text)
        items = root.findall(".//item")
        articles = []
        for item in items:
            title = item.findtext("title") or ""
            articles.append({
                "title": title,
                "source": "Economic Times",
                "url": item.findtext("link") or "",
                "published": item.findtext("pubDate") or ""
            })
            if len(articles) >= max_articles:
                break
        return articles if articles else [{"title": f"{company} stock in focus", "source": "FinVise", "url": "", "published": ""}]
    except:
        return [{"title": f"{company} stock in focus", "source": "FinVise", "url": "", "published": ""}]

Writing news_fetcher.py


In [4]:
%%writefile ai_summary.py
import os
import requests

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

def generate_summary(stock_data, news, ticker="STOCK"):
    name  = stock_data.get("name", ticker)
    price = stock_data.get("price", "N/A")
    open_ = stock_data.get("open", "N/A")
    high  = stock_data.get("high", "N/A")
    low   = stock_data.get("low", "N/A")
    news_titles = ", ".join([n["title"] for n in news[:3]])

    if GROQ_API_KEY:
        try:
            headers = {
                "Authorization": f"Bearer {GROQ_API_KEY}",
                "Content-Type": "application/json"
            }
            payload = {
                "model": "llama3-8b-8192",
                "messages": [{"role": "user", "content": f"Write a 90-second beginner-friendly video script for {name} stock. Price: {price}, Open: {open_}, High: {high}, Low: {low}. News: {news_titles}. Use sections: [0-10 sec | HOOK], [10-30 sec | SNAPSHOT], [30-60 sec | HAPPENING], [60-80 sec | TAKEAWAY], [80-90 sec | CTA]."}],
                "max_tokens": 600
            }
            resp = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload, timeout=20)
            return resp.json()["choices"][0]["message"]["content"].strip()
        except:
            pass

    return f"""[0-10 sec | HOOK]
Did you check {name} today? Here is your 90 second brief!

[10-30 sec | SNAPSHOT]
{name} is trading at Rs {price}. Opened at Rs {open_}, high of Rs {high}, low of Rs {low}.

[30-60 sec | HAPPENING]
Recent news: {news_titles}.

[60-80 sec | TAKEAWAY]
For beginners, focus on fundamentals not just daily price movement.

[80-90 sec | CTA]
Subscribe for daily stock briefs and always do your own research!"""

Writing ai_summary.py


In [5]:
%%writefile video_generator.py
import re, textwrap
from pathlib import Path
from gtts import gTTS
from moviepy.editor import AudioFileClip, ImageClip, concatenate_videoclips
from PIL import Image, ImageDraw, ImageFont
import numpy as np

TMP = Path("/tmp/finvise")
TMP.mkdir(exist_ok=True)
W, H = 1280, 720

def make_frame(title, body, idx):
    img = Image.new("RGB", (W, H), (10, 14, 26))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),(W,6)], fill=(0,255,135))
    try:
        font_big = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 36)
        font_med = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 28)
        font_sm  = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 18)
    except:
        font_big = font_med = font_sm = ImageFont.load_default()
    draw.text((60, 40), f"[{idx}] {title}", font=font_big, fill=(0,255,135))
    draw.line([(60,90),(W-60,90)], fill=(31,41,55), width=1)
    lines = textwrap.wrap(body, width=60)
    for i, line in enumerate(lines[:10]):
        draw.text((60, 110 + i*46), line, font=font_med, fill=(249,250,251))
    draw.rectangle([(0,H-50),(W,H)], fill=(17,24,39))
    draw.text((60, H-35), "FinVise AI - Indian Stock Intelligence", font=font_sm, fill=(107,114,128))
    return np.array(img)

def parse_sections(script):
    parts = [s.strip() for s in re.split(r'\n\n+', script) if s.strip()]
    return parts if parts else [script]

def create_video(script, ticker="STOCK"):
    try:
        audio_path = str(TMP / "voice.mp3")
        gTTS(text=script, lang="en", tld="co.in").save(audio_path)
        audio = AudioFileClip(audio_path)
        parts = parse_sections(script)
        dur_each = audio.duration / len(parts)
        labels = ["HOOK","SNAPSHOT","HAPPENING","TAKEAWAY","CTA"]
        clips = [ImageClip(make_frame(labels[i] if i < len(labels) else f"PART {i+1}", part[:300], i+1)).set_duration(dur_each) for i, part in enumerate(parts)]
        video = concatenate_videoclips(clips, method="compose").set_audio(audio)
        out = str(TMP / f"{ticker}.mp4")
        video.write_videofile(out, fps=24, codec="libx264", audio_codec="aac", logger=None)
        audio.close(); video.close()
        return out
    except Exception as e:
        print("Video error:", e)
        return None

Writing video_generator.py


In [6]:
%%writefile app.py
import streamlit as st
from stock_data import get_stock_data
from news_fetcher import get_news
from ai_summary import generate_summary
from video_generator import create_video

st.title("FinVise AI - Indian Stock Intelligence")
ticker = st.text_input("Enter NSE Stock Ticker (Example: RELIANCE)")

if st.button("Analyze Stock"):
    if not ticker:
        st.warning("Please enter a ticker!")
    else:
        ticker = ticker.strip().upper()
        with st.spinner("Fetching stock data..."):
            stock = get_stock_data(ticker, "NSE")
        st.subheader("Stock Data")
        st.write(stock)
        with st.spinner("Fetching news..."):
            news = get_news(ticker)
        st.subheader("Latest News")
        for n in news:
            st.write("*", n["title"])
        with st.spinner("Generating AI script..."):
            script = generate_summary(stock, news, ticker)
        st.subheader("Generated Script")
        st.write(script)
        with st.spinner("Generating video..."):
            video_path = create_video(script, ticker)
        if video_path:
            st.video(video_path)
        else:
            st.error("Video generation failed.")

Writing app.py


In [8]:
!ngrok config add-authtoken 3B1FMv2N4NdY9OI73ysnUUrephn_3idTHfzFiPGMT9521Muks

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [14]:
!pkill -f ngrok
!pkill -f streamlit
import time
time.sleep(4)

import subprocess
from pyngrok import ngrok

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(6)

url = ngrok.connect(8501)
print("YOUR APP LINK:", url)

YOUR APP LINK: NgrokTunnel: "https://tetragonal-subcallosal-elana.ngrok-free.dev" -> "http://localhost:8501"


In [15]:
import subprocess, time
from pyngrok import ngrok

!pkill -f streamlit
time.sleep(2)

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(6)

ngrok.kill()
url = ngrok.connect(8501)
print("YOUR APP LINK:", url)

YOUR APP LINK: NgrokTunnel: "https://tetragonal-subcallosal-elana.ngrok-free.dev" -> "http://localhost:8501"
